# 02 · DEAP ile Genetik Programlama ve Hızlı Vekil Uygunluk (Aşama 2)

**Araştırma sorusu (RQ2).** Altı normalize öznitelikten oluşan sembolik bir skor $S_i=T_j(\mathbf{f}_i)$, tabakalı Top-K ile seçilen
1.000 örnekle yapılan **ucuz bir mikro-SFT** sonrasında, bağımsız bir doğrulama kümesinde daha düşük çapraz entropi sağlayacak biçimde
evrimleştirilebilir mi?

**Hipotez.** GP'nin bulduğu $f_{GP}^*$, yalnızca IFD'ye göre seçimden ($\text{Baseline}$) ve rastgele seçimden daha düşük $\mathcal{L}_{val}$ üretir.
(Bu defter **vekil** ölçekte yalnızca formülü bulur; hedef modeldeki etki defter 03–04'te sınanır.)

## 2.1 Yöntem

**Gösterim.** Birey $T_j$: önek (prefix) ağacı; terminaller $\{L_x,L_y,PPL,IFD,H_y,D_{div}\}\cup\mathcal{U}[-1,1]$ (geçici rastgele sabit),
fonksiyonlar $\{+,-,\times,\ \text{pdiv},\ \text{plog},\ \text{psqrt},\ \max,\ \min\}$.

| Korumalı operatör | Tanım |
|---|---|
| `pdiv(a,b)` | $1$ eğer $\lvert b\rvert<10^{-6}$, aksi hâlde $a/b$ |
| `plog(a)` | $\log(\lvert a\rvert+10^{-6})$ |
| `psqrt(a)` | $\sqrt{\lvert a\rvert}$ |

**Önbellekli tabakalı vekil uygunluk.** Her birey için:
1. $S_i$ tüm havuzda vektörize hesaplanır.
2. **Tabakalı Top-K:** 20 kümenin her birinden en yüksek $S_i$'li 50 örnek → $\lvert D^{j}_{aday}\rvert = 1000$.
3. **Önbellek:** seçilen indekslerin sıralı `tuple`'ının özeti (SHA-1) `fitness_cache`'te aranır; varsa GPU eğitimi **atlanır**.
4. **Hızlı mikro-SFT:** `Qwen2.5-0.5B` 4-bit QLoRA modeli GPU'ya **bir kez** yüklenir; her yeni bireyde LoRA ağırlıkları
   `reset_lora_parameters` ile başlangıca döndürülür ($A\sim$ Kaiming-uniform, $B=0$) ve 60 adım (toplu 16) eğitilir.
5. **Uygunluk (minimize):**
$$\text{Fitness}(T_j)=\mathcal{L}_{val}(T_j)+0.001\cdot\lvert T_j\rvert$$
   ikinci terim *bloat* (gereksiz ağaç büyümesi) cezasıdır; $\lvert T_j\rvert$ düğüm sayısı.

| Evrim parametresi | Değer |
|---|---|
| Popülasyon $P$ / nesil $G$ | 16 / 10 |
| Başlangıç | `genHalfAndHalf`, derinlik 1–4 |
| Seçilim | Turnuva, $k_{tour}=3$ |
| Elitizm | `HallOfFame(2)` — en iyi 2 birey her nesle taşınır |
| Çaprazlama / Mutasyon | `cxOnePoint` ($p_c=0.7$) / `mutUniform` ($p_m=0.2$) |
| Ek güvenlik | `staticLimit(height ≤ 8)` — üstel ağaç büyümesini engeller |

> **Sınırlılık (kayıt).** $P\times G$ küçük bir bütçedir; bulunan formül *küresel* optimum olmayabilir. Evrim tek tohumla koşulur;
> kararlılık için `SEEDS` ile birden çok koşu yapılması önerilir (bkz. 2.5).

In [ ]:
# --- Ortam kurulumu (Colab + yerel) ---------------------------------------------------
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/gp-stratified-sft-selection")      # <- proje klasörünüzün Drive yolu
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["PROJECT_ROOT"] = str(PROJECT)                  # çıktılar PROJECT/outputs altına yazılır
sys.path.insert(0, str(PROJECT / "src"))

%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from analysis import set_academic_style, show_figure, show_table
set_academic_style()
print("Proje dizini:", PROJECT)

## 2.2 Yapılandırma ve çalıştırma
`DRY_RUN=True` yalnızca mantık testi içindir (GPU eğitimi yerine sahte kayıp). **Gerçek sonuçlar için `False` bırakın.**

In [ ]:
POP, GENS, STEPS = 16, 10, 60
DRY_RUN = False      # True: GPU'suz mantık testi (sahte L_val) -> sonuçlar BİLİMSEL DEĞİLDİR
RUN = True           # False: önceki koşunun çıktıları diskten okunur
FRESH = False        # True: fitness_cache'i silip sıfırdan başla

import json
from deap import gp
from common import GP_DIR, FEAT_DIR, load_json
from gp_evolution import run_gp_evolution, build_pset, to_infix

if RUN:
    res = run_gp_evolution(pop=POP, gens=GENS, steps=STEPS, dry_run=DRY_RUN, fresh=FRESH)
    best, logbook, history, subsets = res["best"], res["logbook"], res["history"], res["subsets"]
else:
    info = load_json(GP_DIR / "best_formula.json")
    best = gp.PrimitiveTree.from_string(info["prefix"], build_pset())
    logbook = pd.read_csv(GP_DIR / "logbook.csv")
    history = pd.read_csv(GP_DIR / "history.csv")
    subsets = {k: np.array(load_json(GP_DIR / "subsets" / f"{k}.json")) for k in ["gp_strat1k", "gp_top1k", "ifd_top1k", "rand1k"]}
info = load_json(GP_DIR / "best_formula.json")
if DRY_RUN:
    display(Markdown("> ⚠️ **DRY-RUN:** uygunluk değerleri sahtedir; bu çıktı yalnızca kod yolunu doğrular."))

## 2.3 Bulgular

### 2.3.1 Keşfedilen formül $f_{GP}^*$

In [ ]:
from analysis import formula_latex
latex = formula_latex(best)
display(Markdown(rf"$$ f_{{GP}}^* \;=\; {latex} $$"))
print("infix :", to_infix(best))
print("prefix:", str(best))
print(f"fitness={best.fitness.values[0]:.4f}  |T|={len(best)}  süre={info.get('minutes', float('nan')):.1f} dk  "
      f"GPU eğitimi={info.get('gpu_trainings')}  önbellek isabeti={info.get('cache_hits')}")

In [ ]:
hof = pd.DataFrame(info["hall_of_fame"])[["fitness", "infix"]]
hof.index = [f"HoF-{i+1}" for i in range(len(hof))]
show_table(hof, "tab_gp_hof", "Onur Listesi (HallOfFame, 2 birey): en iyi bireyler ve uygunlukları.", floatfmt="{:.4f}")

### 2.3.2 Evrimsel yakınsama

In [ ]:
from analysis import plot_gp_convergence, plot_fitness_landscape
lb = pd.DataFrame(logbook)[["gen", "nevals", "min", "med", "avg"]].set_index("gen")
show_table(lb, "tab_gp_logbook", "Nesil bazında uygunluk istatistikleri (min = en iyi). nevals: o nesilde yeniden değerlendirilen birey sayısı.", floatfmt="{:.4f}")
show_figure(plot_gp_convergence(logbook), "fig_gp_convergence", "GP yakınsama eğrisi (düşük = iyi).")
show_figure(plot_fitness_landscape(history), "fig_gp_landscape",
            r"(a) Değerlendirilen her bireyin $\mathcal{L}_{val}$ değeri (önbellek isabetleri GPU maliyeti ödemez); (b) $\mathcal{L}_{val}$'in ağaç boyutuyla ilişkisi.")
hits = int(history["cache_hit"].sum()); tot = len(history)
print(f"Önbellek isabet oranı: {hits}/{tot} = %{100*hits/max(1,tot):.1f}  (GPU mikro-SFT tasarrufu)")

### 2.3.3 Seçilen alt kümelerin profili
`gp_strat1k` (önerilen), `gp_top1k` (kümesiz ablasyon), `ifd_top1k` (baseline), `rand1k` (rastgele).

In [ ]:
from analysis import subset_profile_table, jaccard_table, plot_subset_clusters, plot_score_vs_ifd
pool = pd.read_parquet(FEAT_DIR / "pool.parquet")
prof = subset_profile_table(pool, subsets)
show_table(prof, "tab_subset_profile",
           "Alt kümelerin ham öznitelik ortalamaları, kapsanan küme sayısı ve kümeler arası dağılım entropisi (1 = tüm kümelere eşit dağılım).", floatfmt="{:,.3f}")
show_table(jaccard_table(subsets), "tab_subset_jaccard", "Alt kümeler arası Jaccard benzerliği (örtüşme).", floatfmt="{:.3f}")
show_figure(plot_subset_clusters(pool, subsets), "fig_subset_clusters",
            "Alt kümelerin anlamsal kümelere dağılımı. Tabakalı seçim her kümeden tam 50 örnek alır; kümesiz seçimler birkaç kümede yoğunlaşabilir.")
scores = np.load(GP_DIR / "gp_scores.npy")
show_figure(plot_score_vs_ifd(pool, scores, subsets), "fig_gp_vs_ifd", r"$f_{GP}^*$ skorunun IFD ile ilişkisi; seçilen örnekler vurgulanmıştır.")

## 2.4 Doğrulama: aynı alt küme gerçekten aynı özet (hash) üretiyor mu?
Önbellek doğruluğu, indeks dizisinin sıra bağımsız (sıralı) özetlenmesine bağlıdır.

In [ ]:
from gp_evolution import subset_key
a = np.array([5, 1, 9, 3]); b = np.array([1, 3, 5, 9])
assert subset_key(np.sort(a)) == subset_key(np.sort(b)) and subset_key(np.sort(a)) != subset_key(np.sort(np.array([1, 3, 5, 10])))
assert all(len(v) == 1000 for v in subsets.values()), "Her alt küme tam 1000 örnek içermeli."
assert pool.iloc[subsets["gp_strat1k"]]["cluster"].value_counts().eq(50).all(), "gp_strat1k her kümeden tam 50 örnek içermeli."
print("✔ hash tutarlı  ✔ alt küme boyutları 1000  ✔ gp_strat1k = 20 × 50")

## 2.5 İsteğe bağlı: tohum duyarlılığı
Bir formülün tesadüfi olmadığını göstermek için farklı tohumlarla evrimi tekrarlayıp en iyi uygunlukların dağılımını raporlayın
(`SEEDS` hücresini çalıştırmak her tohum için ≈ GP süresi kadar GPU zamanı alır; `fitness_cache` ortak olduğundan tekrarlayan alt kümeler ücretsizdir).

In [ ]:
RUN_SEEDS = False
SEEDS = [1, 2, 3]
if RUN_SEEDS:
    import common, random
    rows = []
    for s in SEEDS:
        common.set_seed(s); random.seed(s); np.random.seed(s)
        r = run_gp_evolution(pop=POP, gens=GENS, steps=STEPS, dry_run=DRY_RUN)
        rows.append({"tohum": s, "en iyi uygunluk": r["best"].fitness.values[0], "boyut": len(r["best"]), "formül": to_infix(r["best"])})
    show_table(pd.DataFrame(rows).set_index("tohum"), "tab_gp_seeds", "Farklı tohumlarla GP'nin en iyi uygunluğu.", floatfmt="{:.4f}")

## 2.6 Tartışma *(deney sonrası doldurulacak)*
1. $f_{GP}^*$ hangi öznitelikleri kullanıyor? Formülün işaretleri (ör. IFD artıyorsa skor artıyor mu?) literatürle (Cherry-Tagging/IFD) tutarlı mı?
2. Yakınsama: son nesillerde kazanç doyuyor mu? Önbellek isabeti, evrimin erken yakınsadığını (aynı alt kümelerin tekrar seçildiğini) gösteriyor mu?
3. `gp_strat1k` ile `gp_top1k` arasındaki küme entropisi farkı, tabakalamanın çeşitliliğe katkısını nicel olarak gösteriyor mu?
4. **Tehditler:** (i) vekil 0.5B ≠ hedef 3B; (ii) 60 adımlık mikro-SFT gürültülüdür — $\mathcal{L}_{val}$ farkları küçükse seçim büyük ölçüde şans olabilir; (iii) tek tohum; (iv) $D_{val}$ 250 örnektir.

**Sonraki adım:** `03_target_sft_and_alignment` — alt kümelerle 3B hedef modelde 5 SFT ve ardından GRPO/DPO/R-DPO.